# GOMD on Google Colab

Runs GOMD (GPU-accelerated Open MD) from your Google Drive, on a Colab GPU.

```
My Drive/GOMD/code/GOMD/
├── gomd/        engine
├── pkg/         reference package (needed for the one-time setup of every run)
├── md-input/    ff.dat  coords.inp  connectivity.dat  mdrun.params
└── md-output/   logs, comparison tables and the run history are written here
```

**Runtime → Change runtime type → GPU**, then run the cells in order. Cells 1–3 check the setup; cell 4 holds every run setting; cell 5 is only needed for UMA.

GOMD evaluates the polarization SCF, Ewald and bonded terms in fp64. A100 / H100 have fast fp64; **T4 and L4 have very little** (1/32–1/64 of their fp32 rate), so polarizable runs there are much slower than the 56 ms/step measured on the V100. Classical runs are less affected.

Nothing here imports JAX in the notebook itself: every GPU job runs in a subprocess, so the notebook never holds GPU memory that GOMD or UMA need.

In [1]:
#@title 1. Mount Drive and find the project
PROJECT_IN_DRIVE = r'GOMD\code\GOMD'   #@param {type:"string"}
from google.colab import drive
drive.mount('/content/drive')
import os
rel = PROJECT_IN_DRIVE.replace('\\', '/').strip('/')
for base in ('/content/drive/MyDrive', '/content/drive/My Drive'):
    if os.path.isdir(os.path.join(base, rel)):
        WORKING_DIR = os.path.join(base, rel)
        DRIVE_ROOT = base
        break
else:
    raise FileNotFoundError(f'{rel} not found under /content/drive/MyDrive — check PROJECT_IN_DRIVE')
print('project:', WORKING_DIR)

Mounted at /content/drive
project: /content/drive/MyDrive/GOMD/code/GOMD


In [2]:
#@title 2. Check the folder layout
import glob
need = ['gomd/main_scan.py', 'gomd/thermostat.py', 'gomd/mlip/server.py', 'gomd/selftest.py',
        'gomd/profile_step.py', 'pkg/main.py', 'pkg/forces.py',
        'md-input/ff.dat', 'md-input/coords.inp', 'md-input/connectivity.dat', 'md-input/mdrun.params']
missing = [p for p in need if not os.path.exists(os.path.join(WORKING_DIR, p))]
if missing:
    raise FileNotFoundError(f'missing under {WORKING_DIR}: {missing}\n'
                            '(gomd/thermostat.py and gomd/mlip/ come with the 2026-11-16 release, '
                            'gomd/profile_step.py with the 2026-11-17 one)')
os.makedirs(os.path.join(WORKING_DIR, 'md-output'), exist_ok=True)
n_pkg = len(glob.glob(os.path.join(WORKING_DIR, 'pkg', '*.py')))
print(f'layout OK — pkg/ has {n_pkg} .py modules (expected 20)')

# torsion-force fix of 2026-11-09 must be in both trees
fixed = {'gomd/forces/bonded.py': 'sign = -jnp.sum(rij * n', 'pkg/forces.py': 'sign = -(rij[0]*n[0]'}
for f, s in fixed.items():
    ok = s in open(os.path.join(WORKING_DIR, f)).read()
    print(f'torsion fix in {f}: {"yes" if ok else "NO — this copy predates the 2026-11-09 fix"}')

rows = [l.split() for l in open(os.path.join(WORKING_DIR, 'md-input/mdrun.params'))
        if l.strip() and not l.lstrip().startswith('*')]
flags = [t.strip('.').lower() == 'true' or t.upper() == 'T' for t in rows[0][:3]]
print('mdrun.params ensemble:', [n for n, f in zip(('NVE', 'NVT', 'NPT'), flags) if f],
      '| dt', rows[1][2], 'fs | T', rows[2][0], 'K | Num_NHC', rows[2][1], '| OmegaQ', rows[2][2], '1/fs')

layout OK — pkg/ has 20 .py modules (expected 20)
torsion fix in gomd/forces/bonded.py: yes
torsion fix in pkg/forces.py: yes
mdrun.params ensemble: ['NVT'] | dt 0.5 fs | T 296.15 K | Num_NHC 1 | OmegaQ 0.01 1/fs


In [3]:
#@title 3. GPU and Python packages (checked in a subprocess)
#@markdown Needs a **GPU runtime** (Runtime → Change runtime type → A100, L4 or T4). A CPU or TPU
#@markdown runtime has no NVIDIA GPU; TPUs cannot run GOMD's fp64 terms. ALLOW_CPU runs on the CPU
#@markdown anyway, for testing the setup only (a polarizable step takes seconds).
ALLOW_CPU = False   #@param {type:"boolean"}
import subprocess, sys, re, os, shutil, glob
smi_path = shutil.which('nvidia-smi') or next(
    (p for p in ('/usr/bin/nvidia-smi', '/opt/bin/nvidia-smi', '/usr/local/nvidia/bin/nvidia-smi')
     if os.path.exists(p)), None)
HAVE_GPU, GPU, DRIVER_CUDA = False, 'none (CPU)', 0.0
if smi_path:
    smi = subprocess.run([smi_path], capture_output=True, text=True)
    if smi.returncode == 0:
        HAVE_GPU = True
        GPU = subprocess.run([smi_path, '--query-gpu=name,memory.total,driver_version',
                              '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
        m = re.search(r'CUDA Version:\s*([\d.]+)', smi.stdout)
        DRIVER_CUDA = float(m.group(1)) if m else 0.0
if HAVE_GPU:
    print('GPU:', GPU, '| driver supports CUDA', DRIVER_CUDA)
    if any(k in GPU for k in ('T4', 'L4')):
        print('note: weak fp64 on this GPU — polarizable runs will be slow; prefer A100/H100 for production')
else:
    kind = 'a TPU' if (os.path.exists('/dev/accel0') or glob.glob('/dev/vfio/[0-9]*')
                       or 'COLAB_TPU_ADDR' in os.environ) else 'a CPU-only'
    msg = (f'This is {kind} runtime: no NVIDIA GPU (nvidia-smi not found or not working).\n'
           'Fix: Runtime → Change runtime type → Hardware accelerator: A100 GPU (or L4 / T4) → Save,\n'
           'then run the notebook again from cell 1 (changing the runtime restarts it and unmounts Drive).\n'
           'If no GPU is offered, the Colab GPU quota is used up for now.')
    if not ALLOW_CPU:
        raise SystemExit(msg)
    print(msg + '\nALLOW_CPU is set: continuing on the CPU, for testing only.')

probe = ("import jax, numpy, scipy, numba; "
         "print('jax', jax.__version__, '| backend', jax.default_backend(), '|', jax.devices()); "
         "print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| numba', numba.__version__)")
r = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])
JAX_ON_GPU = "backend gpu" in r.stdout
if HAVE_GPU and not JAX_ON_GPU:
    print('JAX does not see the GPU. Set INSTALL_JAX_CUDA = True below and rerun this cell.')
INSTALL_JAX_CUDA = False   #@param {type:"boolean"}
if INSTALL_JAX_CUDA and HAVE_GPU and not JAX_ON_GPU:
    extra = 'cuda13' if DRIVER_CUDA >= 13 else 'cuda12'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', f'jax[{extra}]'], check=True)
    r = subprocess.run([sys.executable, '-c', probe], capture_output=True, text=True)
    print(r.stdout or r.stderr[-2000:])

GPU: NVIDIA L4, 23034 MiB, 580.82.07 | driver supports CUDA 13.0
note: weak fp64 on this GPU — polarizable runs will be slow; prefer A100/H100 for production
jax 0.11.1 | backend gpu | [CudaDevice(id=0)]
numpy 2.4.6 | scipy 1.16.3 | numba 0.64.0



In [4]:
#@title 4. Run settings
FORCE_FIELD   = 'polarizable'  #@param ['polarizable', 'classical']
ENSEMBLE      = 'auto'         #@param ['auto', 'nve', 'nvt']
STEPS         = 20000            #@param {type:"integer"}
PRINT_EVERY   = 50             #@param {type:"integer"}
REBUILD_EVERY = 100            #@param {type:"integer"}
#@markdown Input: `md-input`, or a restart folder from an earlier run (`md-output/gomd_<...>_restart`) to continue it.
#@markdown Every run writes its final state to `md-output/<log name>_restart/`; RESTART_EVERY > 0 also rewrites it during the run.
INPUT_DIR     = 'md-input'     #@param {type:"string"}
RESTART_EVERY = 10000          #@param {type:"integer"}
#@markdown The neighbor list is also rebuilt whenever a site drifts more than skin/2 − 0.01 Å (0.49 Å). To get the
#@markdown old fixed-interval behaviour, put `--no-drift-rebuild` in EXTRA_ARGS; the log then reports any pairs the stale list missed.
#@markdown Tolerances, blank = defaults: SHAKE 1e-12 (the reference's 1e-6 makes the total energy ring by ±4 kcal/mol
#@markdown step to step); SCF blank = mdrun.params (1e-4, which left Jacobi's E_pol 19 kcal/mol unconverged).
#@markdown Put `--reference-parity` in EXTRA_ARGS to reproduce the reference pkg exactly (with SCF_SOLVER = jacobi, SCF_GUESS = cold).
SHAKE_TOL     = ''             #@param {type:"string"}
TOLPOL        = '1e-8'         #@param {type:"string"}
#@markdown SCF solver: `pcg` + `predict` is the default (2,000-step NVE: same energy conservation as Jacobi at 1e-8,
#@markdown with 2.3 instead of 10 T·μ products per step). `jacobi` + `cold` is the reference iteration.
SCF_SOLVER    = 'pcg'          #@param ['pcg', 'jacobi']
SCF_GUESS     = 'predict'      #@param ['predict', 'cold']
#@markdown ML potential: `uma` needs cell 5; `mock` is a built-in stand-in for testing.
MLIP          = 'uma'         #@param ['none', 'uma', 'mock']
PROPAGATE     = 'pff'          #@param ['pff', 'mlip']
COMPARE_EVERY = 50             #@param {type:"integer"}
MLIP_MODEL    = 'uma-s-1p2p1'  #@param ['uma-s-1p2p1', 'uma-s-1p2', 'uma-s-1p1', 'uma-m-1p1']
MLIP_TASK     = 'omol'         #@param ['omol', 'omc', 'omat']
MLIP_SETTINGS = 'high-precision'      #@param ['default', 'high-precision']
EXTRA_ARGS    = ''             #@param {type:"string"}
RUN_LOCAL     = True           #@param {type:"boolean"}
#@markdown RUN_LOCAL copies the code to /content first (Drive is slow for imports and numba's cache); logs are always written to md-output on Drive.
if PROPAGATE == 'mlip' and MLIP == 'none':
    raise ValueError('PROPAGATE = mlip needs MLIP = uma or mock')
if not os.path.exists(os.path.join(WORKING_DIR, INPUT_DIR, 'coords.inp')):
    raise FileNotFoundError(f'{INPUT_DIR}/coords.inp not found under {WORKING_DIR}')
print('settings OK')

settings OK


In [5]:
#@title 5. UMA setup (only for MLIP = 'uma'): fairchem in its own environment + Hugging Face access
#@markdown UMA runs in a separate process from its own virtual environment, so PyTorch and fairchem
#@markdown never share a Python with JAX. Model files are cached on Drive (one download).
#@markdown Needs a Hugging Face token with access to facebook/UMA, stored as the Colab secret **HF_TOKEN** (key icon, left).
UMA_ENV = '/content/uma_env'
UMA_CACHE_ON_DRIVE = False   #@param {type:"boolean"}
import subprocess, os, sys
UMA_PY = os.path.join(UMA_ENV, 'bin', 'python')
MLIP_ENV = {}
if MLIP != 'uma':
    print('MLIP is not uma — nothing to do')
else:
    try:
        from google.colab import userdata
        MLIP_ENV['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        raise SystemExit(f'Colab secret HF_TOKEN not available ({exc}). Add it (key icon) with a token '
                         'that has access to https://huggingface.co/facebook/UMA, and allow notebook access.')
    if UMA_CACHE_ON_DRIVE:
        MLIP_ENV['FAIRCHEM_CACHE_DIR'] = os.path.join(DRIVE_ROOT, 'GOMD', 'fairchem_cache')
        os.makedirs(MLIP_ENV['FAIRCHEM_CACHE_DIR'], exist_ok=True)
    def sh(cmd):
        print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)
    check = ("import torch, fairchem.core as fc; "
             "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| fairchem', fc.__version__)")
    have = os.path.exists(UMA_PY) and subprocess.run([UMA_PY, '-c', check], capture_output=True).returncode == 0
    if not have:
        sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
        sh(['uv', 'venv', '-q', '--python', sys.executable, UMA_ENV])
        # fairchem-core pins torch ~= 2.13; its PyPI wheel is built for CUDA 13 (driver >= 580).
        # On an older driver, take torch from a CUDA 12 index first.
        if DRIVER_CUDA < 13:
            for cu in ('cu129', 'cu128', 'cu126'):
                r = subprocess.run(['uv', 'pip', 'install', '-q', '--python', UMA_PY, 'torch~=2.13.0',
                                    '--index-url', f'https://download.pytorch.org/whl/{cu}'])
                if r.returncode == 0:
                    print('torch from', cu); break
        sh(['uv', 'pip', 'install', '-q', '--python', UMA_PY, 'fairchem-core'])
    r = subprocess.run([UMA_PY, '-c', check], capture_output=True, text=True)
    print(r.stdout or r.stderr[-3000:])
    if 'cuda True' not in r.stdout:
        print('WARNING: PyTorch in the UMA environment does not see the GPU; UMA would run on the CPU.')
    # download (or find in the cache) the model once, checking Hugging Face access
    dl = (f"from fairchem.core import pretrained_mlip as p; "
          f"print('model file:', p.pretrained_checkpoint_path_from_name('{MLIP_MODEL}'))")
    r = subprocess.run([UMA_PY, '-c', dl], env=dict(os.environ, **MLIP_ENV), capture_output=True, text=True)
    print(r.stdout or r.stderr[-3000:])
    if r.returncode != 0:
        raise SystemExit('Model download failed — check that HF_TOKEN has access to facebook/UMA')

$ /usr/bin/python3 -m pip install -q uv
$ uv venv -q --python /usr/bin/python3 /content/uma_env
$ uv pip install -q --python /content/uma_env/bin/python fairchem-core
torch 2.13.0+cu130 | cuda True | fairchem 2.26.0

model file: /root/.cache/fairchem/models--facebook--UMA/snapshots/1f43bb7ff24084b72e8253f9e16fbac19320adae/checkpoints/uma-s-1p2p1.pt



In [ ]:
#@title 6. Run GOMD
import shutil, subprocess, sys, time, os
RUN_DIR = '/content/gomd_run' if RUN_LOCAL else WORKING_DIR
if RUN_LOCAL:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    shutil.copytree(WORKING_DIR, RUN_DIR,
                    ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'md-output', '.ipynb_checkpoints'))
OUT_DIR = os.path.join(WORKING_DIR, 'md-output')
stamp = time.strftime('%Y%m%d-%H%M%S')
tag = f"{FORCE_FIELD}_{ENSEMBLE}" + (f"_{MLIP}-{PROPAGATE}" if MLIP != 'none' else '')
LOG = os.path.join(OUT_DIR, f'gomd_{tag}_{stamp}.out')
# md-input is in the local copy; a restart folder lives in md-output on Drive (not copied)
INPUT = './md-input' if INPUT_DIR.strip('/') == 'md-input' else os.path.join(WORKING_DIR, INPUT_DIR)
cmd = [sys.executable, '-m', 'gomd.main_scan',
       '--input-dir', INPUT, '--ref-dir', './pkg',
       '--steps', str(STEPS), '--print-every', str(PRINT_EVERY),
       '--rebuild-every', str(REBUILD_EVERY), '--ensemble', ENSEMBLE, '--out', LOG,
       '--restart-every', str(RESTART_EVERY)]
if FORCE_FIELD == 'polarizable':
    cmd += ['--polarizable', '--scf-solver', SCF_SOLVER, '--scf-guess', SCF_GUESS]
if SHAKE_TOL.strip(): cmd += ['--shake-tol', SHAKE_TOL.strip()]
if TOLPOL.strip():    cmd += ['--tolpol', TOLPOL.strip(), '--scf-max-iter', '300']
if MLIP != 'none':
    cmd += ['--mlip', MLIP, '--propagate', PROPAGATE, '--compare-every', str(COMPARE_EVERY)]
    if MLIP == 'uma':
        cmd += ['--mlip-python', UMA_PY, '--mlip-model', MLIP_MODEL, '--mlip-task', MLIP_TASK,
                '--mlip-settings', MLIP_SETTINGS]
cmd += EXTRA_ARGS.split()
if not globals().get('HAVE_GPU', True) and '--cpu' not in cmd:
    cmd.append('--cpu')                                  # ALLOW_CPU test run (cell 3)
env = dict(os.environ, PYTHONUNBUFFERED='1')
if MLIP != 'none':
    env['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'      # the MLIP server shares the GPU
    env.update(globals().get('MLIP_ENV', {}))
print('running in', RUN_DIR, '\n ', ' '.join(cmd[1:]), '\n')
console = open(LOG.replace('.out', '.console'), 'w')
proc = subprocess.Popen(cmd, cwd=RUN_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end=''); console.write(line); console.flush()
rc = proc.wait(); console.close()
print(f'\nexit code {rc}\nlog:     {LOG}')
RESTART = os.path.splitext(LOG)[0] + '_restart'
if os.path.isdir(RESTART):
    print(f'restart: {RESTART}\n  to continue: INPUT_DIR = {os.path.relpath(RESTART, WORKING_DIR)!r} in cell 4')
CMP = LOG.replace('.out', '.compare.csv')
if os.path.exists(CMP): print('compare:', CMP)

In [ ]:
#@title 7. Summary: energy conservation, speed, run history
import csv, os
def read_log(path):
    cols, rows = None, []
    for l in open(path):
        if l.startswith('#') or not l.strip(): continue
        t = l.split()
        if t[0] == 'step':
            cols = t; continue
        if cols and len(t) == len(cols) and t[0].isdigit():
            try: rows.append({c: float(v) for c, v in zip(cols, t)})
            except ValueError: pass
    return rows
rows = read_log(LOG)
text = open(LOG).read()
dt = float([l.split() for l in open(os.path.join(RUN_DIR, 'md-input', 'mdrun.params'))
            if l.strip() and not l.lstrip().startswith('*')][1][2])
assert len(rows) >= 2, 'need at least two logged steps'
cons = 'H_ext' if 'H_ext' in rows[0] else 'TE'
# skip step-1 start-up transient (input velocities have components along constrained bonds)
use = [r for r in rows if r['step'] >= 2] or rows
r0, r1 = use[0], use[-1]
span = (r1['step'] - r0['step']) * dt
drift = (r1[cons] - r0[cons]) / span if span else float('nan')
ms = [r['ms/step'] for r in rows]
print(f"{len(rows)} samples | dt {dt} fs | {sum(ms)/len(ms):.1f} ms/step (compile time excluded)")
print(f"conserved quantity {cons}: {r0[cons]:.3f} -> {r1[cons]:.3f} kcal/mol over {span:.1f} fs = {drift:+.4f} kcal/mol/fs")
if 'T' in rows[0]: print(f"temperature {rows[0]['T']:.1f} -> {rows[-1]['T']:.1f} K")
hist = os.path.join(OUT_DIR, 'colab_runs.csv')
new = not os.path.exists(hist)
with open(hist, 'a', newline='') as f:
    w = csv.writer(f)
    if new: w.writerow(['stamp', 'gpu', 'force_field', 'ensemble', 'mlip', 'propagate', 'steps',
                        'shake_tol', 'tolpol', 'ms_per_step', 'conserved', 'drift_kcal_mol_fs', 'log'])
    ens_used = 'nvt' if 'ensemble: NVT' in text else 'nve'
    w.writerow([stamp, GPU, FORCE_FIELD, ens_used, MLIP, PROPAGATE, STEPS, SHAKE_TOL, TOLPOL,
                f"{sum(ms)/len(ms):.1f}", cons, f"{drift:+.5f}", os.path.basename(LOG)])
print('appended to', hist)
for l in text.splitlines():
    if l.startswith('# compare summary') or l.startswith('# compare F_rmse') or l.startswith('MLIP:'):
        print(l)

4 samples | dt 0.5 fs | 70.8 ms/step (compile time excluded)
conserved quantity H_ext: -251477.261 -> -251478.570 kcal/mol over 75.0 fs = -0.0174 kcal/mol/fs
temperature 496.8 -> 348.4 K
appended to /content/drive/MyDrive/GOMD/code/GOMD/md-output/colab_runs.csv


In [ ]:
#@title 8. Plots
import matplotlib.pyplot as plt, csv, os
steps = [r['step'] for r in rows]
keys = ['PE', 'KE', cons] + (['T'] if 'T' in rows[0] else [])
fig, ax = plt.subplots(1, len(keys), figsize=(4.2 * len(keys), 3.2))
for a, k in zip(ax, keys):
    a.plot(steps, [r[k] for r in rows], marker='o', ms=3)
    a.set_title(k); a.set_xlabel('step'); a.set_ylabel('K' if k == 'T' else 'kcal/mol')
plt.tight_layout(); plt.show()
if os.path.exists(CMP):
    C = list(csv.DictReader(open(CMP)))
    s = [int(c['step']) for c in C]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].plot(s, [float(c['dE_shift']) for c in C], marker='o'); ax[0].set_title('E_mlip - E_pff (shifted)'); ax[0].set_ylabel('kcal/mol')
    ax[1].plot(s, [float(c['F_rmse']) for c in C], marker='o'); ax[1].set_title('force RMSE'); ax[1].set_ylabel('kcal/mol/Å')
    ax[2].plot(s, [float(c['cos']) for c in C], marker='o'); ax[2].set_title('force cosine')
    for a in ax: a.set_xlabel('step')
    plt.tight_layout(); plt.show()

In [ ]:
#@title 9. (optional) Self-test: forces = -grad E per term, torsion and thermostat vs the reference, MLIP plumbing
#@markdown Runs on the CPU in a subprocess (about 2–4 minutes). Exit code 0 means every check passed.
import subprocess, sys, os
r = subprocess.run([sys.executable, '-m', 'gomd.selftest', '--input-dir', './md-input', '--ref-dir', './pkg'],
                   cwd=globals().get('RUN_DIR', WORKING_DIR),
                   capture_output=True, text=True)
print('\n'.join(l for l in r.stdout.splitlines() if l.startswith('[') or 'checks passed' in l or 'loaded' in l))
if r.returncode: print(r.stderr[-3000:])

system loaded in 83s: nat=8716 ndummy=1316  jax 0.11.1 on cpu
[PASS] force = -dE/dx, pair (vdW + real-space elec): max rel diff 1.9e-15 (limit 1e-08)
[PASS] force = -dE/dx, bond: max rel diff 0.0e+00 (limit 1e-08)
[PASS] force = -dE/dx, bend: max rel diff 1.8e-15 (limit 1e-08)
[PASS] force = -dE/dx, torsion: max rel diff 5.6e-12 (limit 1e-08)
[PASS] force = -dE/dx, Ewald reciprocal: max rel diff 1.3e-15 (limit 1e-08)
[PASS] force = -dE/dx, Ewald exclusion correction: max rel diff 9.0e-06 (limit 0.0001)
[PASS] lone-pair force redistribution = chain rule: max rel diff 2.4e-16
[PASS] torsion GOMD vs reference: force max rel diff 3.9e-14, energy diff 3.4e-13 kcal/mol
[PASS] Nose-Hoover chain half step vs reference (chain 1 and 3): max rel diff 4.3e-15
[PASS] MLIP server round trip (mock): force vs finite difference: max rel diff 1.3e-07; lone-pair sites zero force: True; elements H2880 Li36 C1920 N36 O784 F1672 S72
10/10 checks passed (184s)


In [ ]:
#@title 10. Profile: where the time of one MD step goes on this GPU
#@markdown Runs `gomd.profile_step` in a subprocess (about 5–15 minutes). It reports, in order:
#@markdown 1. fp64 vs fp32 throughput and scatter costs on this GPU;
#@markdown 2. neighbor-list size: pairs in the list vs pairs inside rcut;
#@markdown 3. every force phase timed on its own;
#@markdown 4. the MD step in `lax.scan`, and how far sites drift before a neighbor list goes stale;
#@markdown 5. (polarizable only) Jacobi vs PCG SCF at tolpol 1e-4 … 1e-12.
#@markdown The report is saved to `md-output/profile_*.txt`. It needs cells 1 and 3 only.
PROFILE_FORCE_FIELD = 'polarizable'   #@param ['polarizable', 'classical']
PROFILE_TOLPOL      = '1e-8'          #@param {type:"string"}
PROFILE_SOLVER      = 'pcg'           #@param ['pcg', 'jacobi']
PROFILE_MD_STEPS    = 200             #@param {type:"integer"}
import subprocess, sys, os, time, shutil
PROF_DIR = '/content/gomd_profile'                     # fresh local copy of the code on Drive
shutil.rmtree(PROF_DIR, ignore_errors=True)
shutil.copytree(WORKING_DIR, PROF_DIR, ignore=shutil.ignore_patterns(
    '__pycache__', '*.pyc', 'md-output', '.ipynb_checkpoints', '*.ipynb'))
gpu_name = globals().get('GPU', 'gpu') if globals().get('HAVE_GPU', True) else 'cpu'
gpu_tag = gpu_name.split(',')[0].replace('NVIDIA', '').strip().replace(' ', '-') or 'gpu'
PROFILE = os.path.join(WORKING_DIR, 'md-output',
                       f"profile_{PROFILE_FORCE_FIELD}_{gpu_tag}_{time.strftime('%Y%m%d-%H%M%S')}.txt")
cmd = [sys.executable, '-m', 'gomd.profile_step', '--input-dir', './md-input', '--ref-dir', './pkg',
       '--tolpol', PROFILE_TOLPOL, '--scf-solver', PROFILE_SOLVER,
       '--md-steps', str(PROFILE_MD_STEPS), '--out', PROFILE]
if PROFILE_FORCE_FIELD == 'polarizable': cmd.append('--polarizable')
if not globals().get('HAVE_GPU', True): cmd.append('--cpu')
print('running in', PROF_DIR, '\n ', ' '.join(cmd[1:]), '\n')
proc = subprocess.Popen(cmd, cwd=PROF_DIR, env=dict(os.environ, PYTHONUNBUFFERED='1'),
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
print(f'\nexit code {proc.wait()}\nreport: {PROFILE}')

In [10]:
#@title 11. MLIP-only MD on a lone-pair-free system (md-input2)
#@markdown Runs `gomd.mlip_md` on `md-input2/system.xyz`: the md-input state without its lone-pair sites
#@markdown (7,400 atoms, elements + positions + velocities + cell; no force field, no constraints, ndof = 3N − 3).
#@markdown For UMA, first set MLIP = 'uma' in cell 4 and run cell 5 (fairchem environment + Hugging Face access).
#@markdown FD_CHECK > 0 compares the forces with finite differences of the energy on that many atoms (at the
#@markdown start and at the end). Every run writes `<log>_restart.xyz`; set MLIP_MD_INPUT to it to continue.
MLIP_MD_BACKEND  = 'uma'                  #@param ['uma', 'mock']
MLIP_MD_INPUT    = 'md-output/mlip_md_uma_nvt_20261118-160311_restart.xyz' #@param {type:"string"}
MLIP_MD_ENSEMBLE = 'nve'                  #@param ['nve', 'nvt']
MLIP_MD_STEPS    = 800                    #@param {type:"integer"}
MLIP_MD_DT       = 0.25                    #@param {type:"number"}
MLIP_MD_FD_CHECK = 0                      #@param {type:"integer"}
import subprocess, sys, os, time, shutil
MD2_DIR = '/content/gomd_mlip'                       # local copy of the engine code
shutil.rmtree(MD2_DIR, ignore_errors=True)
shutil.copytree(os.path.join(WORKING_DIR, 'gomd'), os.path.join(MD2_DIR, 'gomd'),
                ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
stamp2 = time.strftime('%Y%m%d-%H%M%S')
LOG2 = os.path.join(WORKING_DIR, 'md-output',
                    f"mlip_md_{MLIP_MD_BACKEND}_{MLIP_MD_ENSEMBLE}_{stamp2}.out")
cmd = [sys.executable, '-m', 'gomd.mlip_md', '--input', os.path.join(WORKING_DIR, MLIP_MD_INPUT),
       '--mlip', MLIP_MD_BACKEND, '--ensemble', MLIP_MD_ENSEMBLE, '--steps', str(MLIP_MD_STEPS),
       '--dt', str(MLIP_MD_DT), '--fd-check', str(MLIP_MD_FD_CHECK), '--print-every', '10',
       '--out', LOG2]
if MLIP_MD_BACKEND == 'uma':
    if not os.path.exists(globals().get('UMA_PY', '/nonexistent')):
        raise SystemExit('UMA environment missing: set MLIP = "uma" in cell 4 and run cell 5 first')
    cmd += ['--mlip-python', UMA_PY, '--mlip-model', MLIP_MODEL, '--mlip-task', MLIP_TASK,
            '--mlip-settings', MLIP_SETTINGS]
env = dict(os.environ, PYTHONUNBUFFERED='1', **globals().get('MLIP_ENV', {}))
print('running', ' '.join(cmd[1:]), '\n')
proc = subprocess.Popen(cmd, cwd=MD2_DIR, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
print(f'\nexit code {proc.wait()}\nlog: {LOG2}\nrestart: {os.path.splitext(LOG2)[0]}_restart.xyz')

running -m gomd.mlip_md --input /content/drive/MyDrive/GOMD/code/GOMD/md-output/mlip_md_uma_nvt_20261118-160311_restart.xyz --mlip uma --ensemble nve --steps 800 --dt 0.25 --fd-check 0 --print-every 10 --out /content/drive/MyDrive/GOMD/code/GOMD/md-output/mlip_md_uma_nve_20261118-163545.out --mlip-python /content/uma_env/bin/python --mlip-model uma-s-1p2p1 --mlip-task omol --mlip-settings high-precision 

# GOMD MLIP MD  2026-11-18 16:35:45  input /content/drive/MyDrive/GOMD/code/GOMD/md-output/mlip_md_uma_nvt_20261118-160311_restart.xyz
# 2002 atoms (H774 Li11 C516 N11 O216 F452 S22), cluster, no PBC (frame 72.9767 x 72.9767 x 72.9767) A, charge 0, spin 1, ndof 6003 (no constraints), dt 0.25 fs, NVE
# MLIP uma ready in 9.2 s: {'backend': 'uma', 'model': 'uma-s-1p2p1', 'task': 'omol', 'settings': 'high-precision', 'device': 'cuda', 'gpu': 'NVIDIA L4', 'fairchem': '2.26.0', 'torch': '2.13.0+cu130', 'load_s': 6.2}
W1118 16:35:56.848000 25063 torch/_logging/_internal.py:1345] [0/0] Profil